In [29]:
import numpy as np
import pandas as pd

import xgboost as xgb

from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error,r2_score
from sklearn.inspection import permutation_importance

In [8]:
df = datasets.fetch_california_housing( as_frame=True) 

X = df.data
y = df.target

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=0)

XGBoost provides many hyperparameters but we will only consider a few of them (see the [XGBoost documentation](https://xgboost.readthedocs.io/en/latest/parameter.html#general-parameters) for an complete overview). 

Note that we will use the [scikit-learn wrapper interface](https://xgboost.readthedocs.io/en/latest/python/python_api.html#module-xgboost.sklearn):


- `objective`: determines the loss function to be used like `reg:squarederror` for regression problems, `binary:logistic` for logistic regression for binary classification or `multi:softmax`  to do multiclass classification using the softmax objectiv (note that there are more options).


- `n_estimators`: Number of gradient boosted trees. Equivalent to number of boosting rounds.


- `max_depth`: Maximum tree depth for base learners.


- `learning_rate`: Boosting learning rate (xgb’s “eta”). Range is [0,1].


- `subsample`: Subsample ratio of the training instance. Setting it to 0.5 means that XGBoost would randomly sample half of the training data prior to growing trees. 


- `colsample_bytree`: Subsample ratio of columns when constructing each tree. Subsampling occurs once for every tree constructed.


- `colsample_bylevel`: Subsample ratio of columns for each level. Subsampling occurs once for every new depth level reached in a tree. Columns are subsampled from the set of columns chosen for the current tree.


- `gamma`: Minimum loss reduction required to make a further partition on a leaf node of the tree. A higher value leads to fewer splits. 


- `reg_alpha`: L1 regularization term on weights (xgb’s alpha). A large value leads to more regularization.


- `reg_lambda`: L2 regularization term on weights (xgb’s lambda).

- `eval_metric` (default is according to objective). Evaluation metrics for validation data, a default metric will be assigned according to objective (`rmse` for regression, and `logloss` for classificationestimator
).


- `random_state`: Random number seed.


- `early_stopping_rounds`: If you have a validation set, you can use early stopping to find the optimal number of boosting rounds. Early stopping requires at least one set in evals. If there’s more than one, it will use the last. Note that we include this parameter in our fit function.


In [11]:
params = {
    "objective": "reg:squarederror",
    "n_estimators":100,
    "max_depth": 4,
    "learning_rate": 0.01,
    "subsample": 0.8,
    "colsample_bytree": 0.9,
    "colsample_bylevel": 0.8,
    "reg_lambda": 0.1,
    "eval_metric": "rmse",
    "random_state": 42,
}

In [12]:
reg = xgb.XGBRegressor(**params)

In [16]:
reg.fit(X_train, 
        y_train, 
        verbose=False,
        eval_set= [(X_train, y_train), (X_test, y_test)]
        )

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=0.8, colsample_bynode=None, colsample_bytree=0.9,
             device=None, early_stopping_rounds=None, enable_categorical=False,
             eval_metric='rmse', feature_types=None, feature_weights=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.01, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=4, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=100, n_jobs=None,
             num_parallel_tree=None, ...)

In [17]:
y_pred = reg.predict(X_test)

In [28]:
np.sqrt(mean_squared_error(y_test, y_pred))

np.float64(0.79710320252536)

In [30]:
r2_score(y_test,y_pred)

0.5376789349577203

## Feature Importance

In [22]:
reg.feature_importances_

array([0.3721851 , 0.0556258 , 0.1713856 , 0.07182278, 0.01031107,
       0.16383499, 0.08391103, 0.07092363], dtype=float32)

In [26]:
# make a pandas data series 
pd.Series(reg.feature_importances_, index=df.feature_names)

MedInc        0.372185
HouseAge      0.055626
AveRooms      0.171386
AveBedrms     0.071823
Population    0.010311
AveOccup      0.163835
Latitude      0.083911
Longitude     0.070924
dtype: float32

In [33]:
# using permutation
result = permutation_importance(
    reg, X_test, y_test, n_repeats=10, random_state=42
)

In [34]:
pd.Series(result.importances_mean, index=df.feature_names)

MedInc        0.549340
HouseAge      0.016327
AveRooms      0.024519
AveBedrms     0.000381
Population    0.000135
AveOccup      0.088587
Latitude      0.050181
Longitude     0.025075
dtype: float64